In [ ]:
!pip install -U torch


In [ ]:
from src.database_builder import get_db_connection
from src.data_fetch import fetch_raw_from_db
from dotenv import load_dotenv
import pandas as pd

In [ ]:
HF_TOKEN = load_dotenv('HF_TOKEN')

In [64]:
conn = get_db_connection()
raw_data = fetch_raw_from_db(conn=conn)

Fetching raw data from Turso...


c:\Users\Alay\Desktop\Work\Otpp_Analysis\src\data_fetch.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'market': pd.read_sql("SELECT * FROM market_data", conn),
c:\Users\Alay\Desktop\Work\Otpp_Analysis\src\data_fetch.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'financials': pd.read_sql("SELECT * FROM financial_filings_raw", conn),
c:\Users\Alay\Desktop\Work\Otpp_Analysis\src\data_fetch.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  'sec': pd.read_sql("SELECT * FROM sec_mda_risk", conn),
c:\Users\Alay\De

In [98]:
df = raw_data['sec']
df['filing_date'] = pd.to_datetime(df['filing_date']).dt.date
df['item_type'] = df['item_type'].replace('RiskFactorsUpdate', 'RiskFactors')

df.head()


,chunk_id,doc_id,ticker,filing_date,item_type,chunk_index,content,sentiment_score,neutral_score
0,MSFT_10Q_20260128_MDA_0,MSFT_10Q_20260128,MSFT,2026-01-28,MDA,0,"Economic Conditions, Challenges, and Risks\nTh...",0.203652,0.693252
1,MSFT_10Q_20260128_MDA_1,MSFT_10Q_20260128,MSFT,2026-01-28,MDA,1,"For the majority of our products, we have the ...",0.918023,0.041764
2,MSFT_10Q_20260128_MDA_2,MSFT_10Q_20260128,MSFT,2026-01-28,MDA,2,More Personal Computing revenue decreased driv...,-0.695678,0.029354
3,MSFT_10Q_20260128_MDA_3,MSFT_10Q_20260128,MSFT,2026-01-28,MDA,3,Prior year net income and diluted EPS were neg...,0.763420,0.053543
4,MSFT_10Q_20260128_MDA_4,MSFT_10Q_20260128,MSFT,2026-01-28,MDA,4,Dynamics 365 revenue grew 19% with growth acro...,0.939139,0.025843


In [99]:
out = (
    df.groupby(["filing_date", "item_type"], as_index=False)
      .agg(content=("content", list))
)
out["filing_date"] = pd.to_datetime(out["filing_date"])


In [122]:
out

,filing_date,item_type,content
0,2018-01-31,MDA,"[Economic Conditions, Challenges, and Risks\nT..."
1,2018-01-31,RiskFactors,[ITEM 1A. RISK FACTORS Our operations and fina...
2,2018-04-26,MDA,"[Economic Conditions, Challenges, and Risks\nT..."
3,2018-04-26,RiskFactors,[ITEM 1A. RISK FACTORS Our operations and fina...
4,2018-08-03,MDA,"[Commercial cloud revenue, which primarily com..."
...,...,...,...
61,2025-07-30,RiskFactors,[ITEM 1A. RISK FACTORS Our operations and fina...
62,2025-10-29,MDA,"[Economic Conditions, Challenges, and Risks\nT..."
63,2025-10-29,RiskFactors,[ITEM 1A. RISK FACTORS Our operations and fina...
64,2026-01-28,MDA,"[Economic Conditions, Challenges, and Risks\nT..."


In [ ]:
out.info()

In [ ]:
out[(out['filing_date'] == '2018-01-31') & (out['item_type'] == 'MDA')]['content'].iloc

0    [Economic Conditions, Challenges, and Risks\nT...
Name: content, dtype: object

In [113]:
mask_q1 = (out['filing_date'] == '2025-10-29') & (out['item_type'] == 'RiskFactors')
q1_mda = out[mask_q1]['content'].iloc[0]
mask_q2 = (out['filing_date'] == '2026-01-28') & (out['item_type'] == 'RiskFactors')
q2_mda = out[mask_q2]['content'].iloc[0]

In [101]:
from sentence_transformers import SentenceTransformer, util
from transformers import BertTokenizer, BertModel

import torch

#Intialize model (check logged as sentance transformer or classification on HF)
model_name = 'yiyanghkust/finbert-tone'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)







Loading weights: 100%|██████████| 199/199 [00:00<00:00, 64773.13it/s]
[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-tone
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [102]:
q1_mda

['Economic Conditions, Challenges, and Risks\nThe markets for software, devices, and cloud-based services are dynamic and highly competitive. Our competitors are developing new software and devices, while also deploying competing cloud-based services for consumers and businesses. The devices and form factors customers prefer evolve rapidly, and influence how users access services in the cloud, and in some cases, the user’s choice of which suite of cloud-based services to use. We must continue to evolve and adapt over an extended time in pace with this changing environment. The investments we are making in infrastructure and devices will continue to increase our operating costs and may decrease our operating margins.\nOur success is highly dependent on our ability to attract and retain qualified employees. We hire a mix of university and industry talent worldwide. Microsoft competes for talented individuals globally by offering an exceptional working environment, broad customer reach, s

In [114]:
def clean_sec_text(text_list):
    return [
        t.replace('\xa0', ' ').replace('\n', ' ').strip() 
        for t in text_list
    ]

q1_mda_clean = clean_sec_text(q1_mda)
q2_mda_clean = clean_sec_text(q2_mda)


In [104]:

def encode_finance_text(text_list):
    # Ensure it's list[str] (already is in your case)
    inputs = tokenizer(
        text_list, 
        padding=True, 
        truncation=True, 
        max_length=512,  # Add this for long SEC texts
        return_tensors="pt"
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = torch.mean(outputs.last_hidden_state, dim=1)
    
    return embeddings

In [115]:
# 2. Simulate Microsoft 10-K Risk Sections (Quarter-over-Quarter)
# Q3: Original text
q3_risk_section = [
    "We face intense competition in all our markets.", # Chunk 0
    "Our cloud computing business depends on data center uptime.", # Chunk 1
    "Fluctuations in foreign exchange rates may affect our results." # Chunk 2
]

# Q4: Updated text (We added a new risk and shifted the old ones)
q4_risk_section = [
    "We face intense competition in all our markets.", # Same as Q3
    "NEW: Emerging AI regulations in Europe may impact our cloud margins.", # BRAND NEW
    "Our cloud computing business depends on data center uptime.", # Shifted from ID 1 to ID 2
    "Exchange rate fluctuations can impact our international revenue." # Paraphrased version of Q3 Chunk 2
]

# 3. Step A: Encode all chunks into Vectors
q3_embeddings = encode_finance_text(q1_mda )
q4_embeddings = encode_finance_text(q2_mda)



In [124]:
out['filing_date'].dt.to_period('Q')


0     2018Q1
1     2018Q1
2     2018Q2
3     2018Q2
4     2018Q3
       ...  
61    2025Q3
62    2025Q4
63    2025Q4
64    2026Q1
65    2026Q1
Name: filing_date, Length: 66, dtype: period[Q-DEC]

In [117]:
# 4. Step B: Semantic Alignment (The "Many-to-One" Search)
print(f"{'New Chunk Text':<60} | {'Best Match Score':<15} | {'Status'}")
print("-" * 90)

drill_down_features = []

for i, q4_vec in enumerate(q4_embeddings):
    # Calculate cosine similarity between THIS new chunk and ALL old chunks
    # This is the "Search" part that ignores position shifts
    cosine_scores = util.cos_sim(q4_vec, q3_embeddings)
    
    # Find the best score and the index of that match
    best_score = torch.max(cosine_scores).item()
    
    status = "REUSE" if best_score > 0.95 else "PARAPHRASE" if best_score > 0.8 else "NEW SIGNAL"
    
    if q2_mda[i]:
        print(f"{q2_mda[i][:59]:<60} | {best_score:<15.4f} | {status}")
        drill_down_features.append(best_score)
    else:
        print(q4_risk_section[i])
# 5. Step C: Generate Features for LightGBM
max_dev = min(drill_down_features) # The "Maximum Surprise"
avg_dev = sum(drill_down_features) / len(drill_down_features)

print("\n--- Model Features for LightGBM ---")
print(f"Minimum Similarity (High Deviation): {max_dev:.4f}")
print(f"Average Document Similarity: {avg_dev:.4f}")

New Chunk Text                                               | Best Match Score | Status
------------------------------------------------------------------------------------------
ITEM 1A. RISK FACTORS Our operations and financial results   | 1.0000          | REUSE
A well-established ecosystem creates beneficial network eff  | 1.0000          | REUSE
Users continue to turn to these devices to perform function  | 1.0000          | REUSE
For all of these reasons, we may not be able to compete suc  | 1.0000          | REUSE
We bear the costs of converting original ideas into softwar  | 1.0000          | REUSE
We are incurring significant costs to build and maintain in  | 1.0000          | REUSE
This could adversely affect our operations, financial condi  | 1.0000          | REUSE
If customers do not perceive our latest offerings as provid  | 1.0000          | REUSE
We have a long-term strategic partnership with OpenAI, and   | 0.9974          | REUSE
It may take longer than expected to r

In [121]:
q2_mda[50]

'The unionization of significant employee populations could result in higher costs and other operational changes necessary to respond to changing conditions and to establish new relationships with worker representatives.'